# 02b: Flow Computation (3D Native)

A focused walkthrough of the primary method — 3D-native scene flow — from a projected
frame pair to its velocity field and a full kinematics dashboard.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from seamless import FlowEstimator, KinematicsAnalyzer
from seamless.config import ThreeDNativeConfig
from seamless.vis.flow_dashboards import plot_flow_field

## Load a frame pair

In [ ]:
full = FlowEstimator.from_projection_h5(
    Path('data/synthetic/ellipsoid_projection.h5'),
    source_file=Path('data/synthetic/ellipsoid.h5'),
)
PAIR = 0
estimator = FlowEstimator(full.frames[PAIR:PAIR + 2],
                          three_d_config=ThreeDNativeConfig())
print(f't={PAIR} -> {PAIR + 1}, uv_res={estimator.frames[0].uv_res},',
      'volume attached:', estimator.frames[0].volume is not None)

## Estimate the 3D velocity field

In [ ]:
field = estimator.estimate('3d_native')[0]
speed = np.linalg.norm(field.v3d, axis=-1)
print('v3d:', field.v3d.shape, '| speed range:',
      round(float(speed.min()), 3), round(float(speed.max()), 3))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(field.frame_t.max_projection, cmap='gray', origin='lower')
ax[0].set_title('max projection'); ax[0].axis('off')
im = ax[1].imshow(speed, cmap='viridis', origin='lower'); ax[1].set_title('|v3d| (speed)')
ax[1].axis('off'); fig.colorbar(im, ax=ax[1]); plt.show()

## Kinematics dashboard

_The 3D autograd kinematics over the full UV grid can take a few minutes on CPU._

In [ ]:
analyzer = KinematicsAnalyzer(estimator.frames, [field])
eul = analyzer.compute_eulerian(field)
print('metrics:', sorted(eul.keys()))

plot_flow_field(field.frame_t.max_projection, field.frame_t.xyz_map_voxel, eul,
                title=f'3D native  (t={PAIR} -> {PAIR + 1})')
plt.show()

## Summary

`FlowEstimator.estimate('3d_native')` trains a 3D FlowMLP on the raw volume and returns
a `FlowField`; `KinematicsAnalyzer.compute_eulerian` derives divergence, curl, normal /
tangential velocity, and the vector Laplacian analytically via autograd, shown here with
the A1-style `plot_flow_field` dashboard.